# 02 Clean Company Entity-Resolution Data
This notebook standardizes the raw company entity-resolution data, creates stable candidate IDs, normalizes left-side input and right-side candidate fields, builds searchable text fields, and writes the cleaned records to `company_er_clean`.

In [0]:
# Clean and normalize company fields for downstream matching.
from pyspark.sql import functions as F

source_table = "workspace.entity_resolution_project.company_er_ready"
target_table = "workspace.entity_resolution_project.company_er_clean"

df = spark.table(source_table)

clean_df = (
    df
    .withColumn(
        "unique_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("input_row_key").cast("string"), F.lit("")),
                F.coalesce(F.col("input_company_name").cast("string"), F.lit("")),
                F.coalesce(F.col("company_name").cast("string"), F.lit("")),
                F.coalesce(F.col("main_country_code").cast("string"), F.lit("")),
                F.coalesce(F.col("main_city").cast("string"), F.lit("")),
                F.coalesce(F.col("website_domain").cast("string"), F.lit(""))
            ),
            256
        )
    )

    # Left side: original input/procurement record
    .withColumn(
        "left_clean_company_name",
        F.upper(F.trim(F.col("input_company_name")))
    )
    .withColumn(
        "left_country_code",
        F.upper(F.trim(F.col("input_main_country_code")))
    )
    .withColumn(
        "left_country",
        F.upper(F.trim(F.col("input_main_country")))
    )
    .withColumn(
        "left_region",
        F.upper(F.trim(F.col("input_main_region")))
    )
    .withColumn(
        "left_city",
        F.upper(F.trim(F.col("input_main_city")))
    )
    .withColumn(
        "left_postcode",
        F.upper(F.trim(F.col("input_main_postcode")))
    )
    .withColumn(
        "left_street",
        F.upper(F.trim(F.col("input_main_street")))
    )

    # Right side: fetched candidate company
    .withColumn(
        "right_clean_company_name",
        F.upper(F.trim(F.col("company_name")))
    )
    .withColumn(
        "right_country_code",
        F.upper(F.trim(F.col("main_country_code")))
    )
    .withColumn(
        "right_country",
        F.upper(F.trim(F.col("main_country")))
    )
    .withColumn(
        "right_region",
        F.upper(F.trim(F.col("main_region")))
    )
    .withColumn(
        "right_city",
        F.upper(F.trim(F.col("main_city")))
    )
    .withColumn(
        "right_postcode",
        F.upper(F.trim(F.col("main_postcode")))
    )
    .withColumn(
        "right_street",
        F.upper(F.trim(F.col("main_street")))
    )

    # Text used later for LLM/explanation, not for blocking as a single identity field
    .withColumn(
        "left_search_text",
        F.concat_ws(
            " ",
            F.coalesce(F.col("input_company_name"), F.lit("")),
            F.coalesce(F.col("input_main_city"), F.lit("")),
            F.coalesce(F.col("input_main_country"), F.lit(""))
        )
    )
    .withColumn(
        "right_search_text",
        F.concat_ws(
            " ",
            F.coalesce(F.col("company_name"), F.lit("")),
            F.coalesce(F.col("company_legal_names"), F.lit("")),
            F.coalesce(F.col("company_commercial_names"), F.lit("")),
            F.coalesce(F.col("short_description"), F.lit("")),
            F.coalesce(F.col("main_city"), F.lit("")),
            F.coalesce(F.col("main_country"), F.lit("")),
            F.coalesce(F.col("website_domain"), F.lit(""))
        )
    )
)

(
    clean_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

display(
    clean_df.select(
        "input_row_key",
        "unique_id",
        "input_company_name",
        "company_name",
        "left_clean_company_name",
        "right_clean_company_name",
        "left_country_code",
        "right_country_code",
        "left_city",
        "right_city",
        "website_domain"
    ).limit(30)
)

In [0]:
# Validate cleaned row counts and unique record counts.
display(
    clean_df.agg(
        F.count("*").alias("rows"),
        F.countDistinct("input_row_key").alias("unique_input_records"),
        F.countDistinct("unique_id").alias("unique_candidate_rows")
    )
)